# 03 · Fine-tune SmolVLA (Steps B1, B3)

One notebook for all three variants: set `CONFIG` below (`v1_rgb` for B1, then `v2_mask` / `v3_grounded` for B3).

1. **Throughput test:** 200 steps without saving, to estimate the full run's time.
2. **Train:** 20k steps from `lerobot/smolvla_base`, backbone frozen. Checkpoints go to `MyDrive/groundedvla/checkpoints/<run>/` every 2k steps. **After a disconnect, re-run every cell from the top**: training resumes from the newest checkpoint.
3. **Checkpoint selection:** 10 rollouts per task on the *validation* demos' initial states for 10k / 15k / 20k (N = 10).
4. **Execution mode (B1 only):** at the selected step, N = 5 / 10 / 25 and temporal ensemble, on the same validation rollouts.
5. **First in-distribution test:** 25 rollouts per task on LIBERO's fixed test states, with videos.

Needs an **A100** (bf16). Upload the dataset first: `D:\groundedvla\datasets\lerobot\gvla_libero` → `MyDrive/groundedvla/datasets/lerobot/gvla_libero`. The raw LIBERO demos (`MyDrive/groundedvla/datasets/libero`, from notebook 00) are used for the validation initial states.

In [ ]:
# ---- edit once ----
REPO_URL = "https://github.com/kaustubhadhe1206/GroundedVLA.git"   # public repo, no token needed
BRANCH = "main"
CONFIG = "configs/v1_rgb.yaml"     # B1: v1_rgb   B3: v2_mask, v3_grounded
WANDB = False                      # True: also log to Weights & Biases (run `wandb login` first)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "/content/GroundedVLA"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
!git log -1 --format="commit %h  %s"

In [ ]:
!apt-get -qq install -y libegl1 libgl1 > /dev/null
%cd /content/GroundedVLA
!git pull --ff-only && git log -1 --format="commit %h  %s"
!bash scripts/colab_install.sh requirements.txt

In [ ]:
import json, yaml
from pathlib import Path
from groundedvla.utils.colab import setup_session, stage_to_local
paths = setup_session()          # env vars (EGL, caches on local disk), Drive folders, LIBERO config

cfg = yaml.safe_load(Path(CONFIG).read_text())
RUN = cfg["name"]
RUN_DIR = Path(paths["checkpoints"]) / RUN
EVAL_DIR = Path(paths["outputs"]) / "eval" / RUN / "smolvla_ckpt"
WANDB_FLAG = "--wandb" if WANDB else ""

# fast local copies (Drive is slow for many small reads)
stage_to_local("datasets/lerobot/gvla_libero")
if (Path(paths["models"]) / "smolvla_base" / "config.json").exists():
    stage_to_local("models/smolvla_base")    # otherwise train.py downloads it to Drive first
print("run:", RUN, "->", RUN_DIR)

## 1. Throughput test (200 steps, nothing saved)

In [ ]:
!python -m groundedvla.train.train --config {CONFIG} --run-name {RUN}_throughput --max-steps 200 --no-save --no-resume

In [ ]:
rows = [json.loads(l) for l in (Path(paths["checkpoints"]) / f"{RUN}_throughput" / "metrics.jsonl").read_text().splitlines()]
speed = [r["steps_per_s"] for r in rows if "steps_per_s" in r][-1]      # last window, after warm-up
mem = max(r.get("gpu_mem_gb", 0) for r in rows)
print(f"{speed:.2f} steps/s at batch {cfg['train']['batch_size']}, peak GPU memory {mem:.1f} GB")
print(f"estimated time for {cfg['train']['steps']} steps: {cfg['train']['steps'] / speed / 3600:.1f} h "
      f"(+ ~1 min per checkpoint and validation)")

## 2. Train (auto-resumes from the newest checkpoint on Drive)

If the data loader is the bottleneck (`data` above ~30% in the log), try `--num-workers 12`.

In [ ]:
!python -m groundedvla.train.train --config {CONFIG} {WANDB_FLAG}

In [ ]:
import matplotlib.pyplot as plt
rows = [json.loads(l) for l in (RUN_DIR / "metrics.jsonl").read_text().splitlines() if l.strip()]
tr = [r for r in rows if "train_loss" in r]
va = [r for r in rows if "val_loss" in r]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].plot([r["step"] for r in tr], [r["train_loss"] for r in tr], label="train")
ax[0].plot([r["step"] for r in va], [r["val_loss"] for r in va], "o-", label="val")
ax[0].set_yscale("log"); ax[0].set_title("flow-matching loss"); ax[0].legend()
ax[1].plot([r["step"] for r in tr], [r["lr"] for r in tr]); ax[1].set_title("learning rate")
ax[2].plot([r["step"] for r in tr], [r["grad_norm"] for r in tr]); ax[2].set_title("grad norm")
for a in ax: a.set_xlabel("step")
plt.tight_layout(); plt.show()
print(json.loads((RUN_DIR / "run_state.json").read_text()))
print(sorted(p.name for p in RUN_DIR.glob("step_*")))

## 3. Checkpoint selection: 10 validation rollouts per task at 10k / 15k / 20k (N = 10)

Resumable: finished episodes are skipped when a cell is re-run.

In [ ]:
SELECT_STEPS = cfg["train"]["select_from_steps"]
for step in SELECT_STEPS:
    ckpt = RUN_DIR / f"step_{step:06d}"
    !python -m groundedvla.eval.run_eval --config {CONFIG} --checkpoint {ckpt} --split val --horizon 10 --tag step_{step:06d}_N10 --no-video

In [ ]:
import pandas as pd

def val_table():
    recs = []
    for f in EVAL_DIR.glob("*/T*_val.jsonl"):
        for l in f.read_text().splitlines():
            if l.strip():
                recs.append(json.loads(l) | {"tag": f.parent.name})
    df = pd.DataFrame(recs)
    t = df.pivot_table(index="tag", columns="task", values="success", aggfunc="mean")
    t["mean"] = t.mean(axis=1)
    t["episodes"] = df.groupby("tag").size()
    return t.sort_index()

table = val_table()
display(table.style.format("{:.0%}", subset=[c for c in table.columns if c != "episodes"]))
sel = table.loc[[f"step_{s:06d}_N10" for s in SELECT_STEPS if f"step_{s:06d}_N10" in table.index]]
BEST_STEP = int(sel["mean"].idxmax().split("_")[1])      # ties -> earliest step
print("selected checkpoint:", BEST_STEP)

## 4. Execution mode (B1 only): N = 5 / 10 / 25 and temporal ensemble at the selected step

Pick the best and write it into `eval.mode` / `eval.execution_horizon` of **all three** configs, so V2 and V3 use the same.

In [ ]:
ckpt = RUN_DIR / f"step_{BEST_STEP:06d}"
for n in [5, 25]:
    !python -m groundedvla.eval.run_eval --config {CONFIG} --checkpoint {ckpt} --split val --horizon {n} --tag step_{BEST_STEP:06d}_N{n} --no-video
!python -m groundedvla.eval.run_eval --config {CONFIG} --checkpoint {ckpt} --split val --mode temporal_ensemble --tag step_{BEST_STEP:06d}_TE --no-video
table = val_table()
display(table.style.format("{:.0%}", subset=[c for c in table.columns if c != "episodes"]))

## 5. First in-distribution test: 25 rollouts per task on LIBERO's fixed test states (videos saved)

Set `MODE` / `N` to the winner of section 4.

In [ ]:
MODE, N = cfg["eval"]["mode"], cfg["eval"]["execution_horizon"]
ckpt = RUN_DIR / f"step_{BEST_STEP:06d}"
tag = f"test_step_{BEST_STEP:06d}_{'TE' if MODE == 'temporal_ensemble' else f'N{N}'}"
!python -m groundedvla.eval.run_eval --config {CONFIG} --checkpoint {ckpt} --split test --mode {MODE} --horizon {N} --episodes 25 --tag {tag}

In [ ]:
!python -m groundedvla.eval.report {EVAL_DIR / tag}

In [ ]:
from IPython.display import Video, display, Markdown
for task in cfg["tasks"]:
    recs = {json.loads(l)["episode_idx"]: json.loads(l) for l in (EVAL_DIR / tag / f"{task}_in_distribution.jsonl").read_text().splitlines() if l.strip()}
    fails = [i for i, r in recs.items() if not r["success"]][:2]
    wins = [i for i, r in recs.items() if r["success"]][:1]
    display(Markdown(f"### {task}: {sum(r['success'] for r in recs.values())}/{len(recs)} success"))
    for i in wins + fails:
        r = recs[i]
        print(f"ep {i}: {'SUCCESS' if r['success'] else 'fail (' + str(r['failure']) + ')'}  steps={r['steps']}")
        display(Video(str(EVAL_DIR / tag / "videos" / f"{task}_in_distribution_ep{i:03d}.mp4"), embed=True, width=512))